# Recursive Trace of a Matrix Product

**Problem:** Given two $n \times n$ matrices $X$ and $Y$, compute $\text{Tr}(XY) = \sum_i (XY)_{ii}$ without forming the full product $XY$.

**Key insight:** Partition both matrices into four $n/2 \times n/2$ blocks:

$$X = \begin{pmatrix} X_0 & X_1 \\ X_2 & X_3 \end{pmatrix}, \quad Y = \begin{pmatrix} Y_0 & Y_1 \\ Y_2 & Y_3 \end{pmatrix}$$

Then:

$$\text{Tr}(XY) = \text{Tr}(X_0 Y_0) + \text{Tr}(X_1 Y_2) + \text{Tr}(X_2 Y_1) + \text{Tr}(X_3 Y_3)$$

Each sub-problem is half the size, and there are 4 of them — giving $O(n^2)$ total work (vs. $O(n^3)$ for computing the full product then taking the trace).

In [1]:
import numpy as np


## Quadrant Splitting

`decouper_matrice` splits a matrix into its four quadrants. This is used by the recursive `Trace` function below.

## Recursive Trace

Base case: for $1 \times 1$ matrices, $\text{Tr}(XY) = x_{00} \cdot y_{00}$.

Recursive case: split into quadrants and sum the four sub-traces as derived above.

## Verification

We compare our recursive trace against `np.trace(X @ Y)` for matrices of size 2 through 16. The difference is at most ~$10^{-14}$ (floating-point precision).

In [2]:

def decouper_matrice(matrice):
  """Découpe une matrice en 4 sous-matrices.

  Args:
    matrice: La matrice à découper.

  Returns:
    Un tuple contenant les 4 sous-matrices.
  """
  lignes, colonnes = matrice.shape
  milieu_lignes = lignes // 2
  milieu_colonnes = colonnes // 2

  sous_matrice1 = matrice[:milieu_lignes, :milieu_colonnes]
  sous_matrice2 = matrice[:milieu_lignes, milieu_colonnes:]
  sous_matrice3 = matrice[milieu_lignes:, :milieu_colonnes]
  sous_matrice4 = matrice[milieu_lignes:, milieu_colonnes:]

  return sous_matrice1, sous_matrice2, sous_matrice3, sous_matrice4

# Exemple d'utilisation
matrice = np.array([[1, 2, 3, 4],
                   [5, 6, 7, 8],
                   [9, 10, 11, 12],
                   [13, 14, 15, 16]])

sous_matrice1, sous_matrice2, sous_matrice3, sous_matrice4 = decouper_matrice(matrice)

print("Sous-matrice 1:\n", sous_matrice1)
print("Sous-matrice 2:\n", sous_matrice2)
print("Sous-matrice 3:\n", sous_matrice3)
print("Sous-matrice 4:\n", sous_matrice4)

Sous-matrice 1:
 [[1 2]
 [5 6]]
Sous-matrice 2:
 [[3 4]
 [7 8]]
Sous-matrice 3:
 [[ 9 10]
 [13 14]]
Sous-matrice 4:
 [[11 12]
 [15 16]]


In [3]:
def Trace(x, y):
  """Calcule la trace du produit de deux matrices de manière récursive."""
  if x.shape == (1, 1):  # Check if x is a 1x1 matrix
    return x[0, 0] * y[0, 0]  # Base case: element-wise multiplication
  sous_x0, sous_x1, sous_x2, sous_x3 = decouper_matrice(x)
  sous_y0, sous_y1, sous_y2, sous_y3 = decouper_matrice(y)
  return (Trace(sous_x0, sous_y0) + Trace(sous_x1, sous_y2) +
          Trace(sous_x2, sous_y1) + Trace(sous_x3, sous_y3))


In [4]:
import numpy as np

# ... (decouper_matrice and Trace functions from previous response) ...

def test_trace(size):
  """Tests the Trace function with matrices of given size."""
  x = np.random.rand(size, size)
  y = np.random.rand(size, size)

  # Calculate trace using our function and NumPy's function
  trace_recursive = Trace(x, y)
  trace_numpy = np.trace(x @ y)  # @ is matrix multiplication

  # Compare the results
  print(f"Size: {size}x{size}")
  print(f"Recursive Trace: {trace_recursive}")
  print(f"NumPy Trace: {trace_numpy}")
  print(f"Difference: {abs(trace_recursive - trace_numpy)}\n")

# Test with different matrix sizes
for size in [2, 4, 8, 16]:
  test_trace(size)

Size: 2x2
Recursive Trace: 1.1261555707257724
NumPy Trace: 1.1261555707257722
Difference: 2.220446049250313e-16

Size: 4x4
Recursive Trace: 4.400820457699346
NumPy Trace: 4.400820457699346
Difference: 0.0

Size: 8x8
Recursive Trace: 19.18435021545447
NumPy Trace: 19.18435021545447
Difference: 0.0

Size: 16x16
Recursive Trace: 67.46689529721921
NumPy Trace: 67.46689529721922
Difference: 1.4210854715202004e-14

